# Construcción del dataset de demografía, salud e ingreso

**Estudiante:** Jorge Adalberto Zuñiga Oliva  
**Matrícula:** 82005 · **Grupo:** 512  
**Materia:** Tópicos de Big Data

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/curso.py").exists())
sys.path.insert(0, str(RAIZ / "scripts"))
from curso import (leer_owid, metadatos, indicador, solo_paises, guardar_figura,
                   guardar_tabla, U1, U3, SALIDA, FUENTES)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 12)

## 1. Extender el dataset demográfico
**Entregable de la tarea:** exactamente ocho columnas: `entity`, `year`, `population`,
`births`, `deaths`, `child_deaths_under_5`, `percentage_health`, `gdp`.
Un segundo archivo agrega esperanza de vida para la práctica de seis predictoras.
Se ejecuta después de `U3_2_Preparacion_dataset.ipynb`.

In [2]:
base = pd.read_csv(SALIDA / "demografia_owid_2000_2022.csv")
salud = indicador("total-healthcare-expenditure-gdp",
    "current_health_expenditure__che__as_percentage_of_gross_domestic_product__gdp__pct", "percentage_health")
pib = indicador("gdp-per-capita-worldbank", "ny_gdp_pcap_pp_kd", "gdp")
vida = indicador("life-expectancy", "life_expectancy_0", "life_expectancy")
print("Filas de entrada:", len(base))
columnas = ["entity", "year", "population", "births", "deaths", "child_deaths_under_5", "percentage_health", "gdp"]
integrado = base.merge(salud.drop(columns="code"), on=["entity", "year"], validate="one_to_one")
print("Tras añadir salud:", len(integrado))
integrado = integrado.merge(pib.drop(columns="code"), on=["entity", "year"], validate="one_to_one")
print("Tras añadir PIB:", len(integrado))
integrado = integrado.dropna(subset=columnas[2:]).sort_values(["entity", "year"]).reset_index(drop=True)
tarea = integrado[columnas].copy()
assert tarea.columns.tolist() == columnas
assert not tarea.duplicated(["entity", "year"]).any()
display(tarea.head())
guardar_tabla(tarea, "dataset_health_gdp_demografia_final.csv", U3 / "data")

Filas de entrada: 5818
Tras añadir salud: 4416
Tras añadir PIB: 4244


,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
0,Afghanistan,2002,21378123.0,1015202.0,239161.0,123215,9.443391,1774.3087
1,Afghanistan,2003,22733053.0,1069727.0,243573.0,123894,8.941258,1815.9282
2,Afghanistan,2004,23560656.0,1095372.0,244435.0,122315,9.808474,1776.9182
3,Afghanistan,2005,24404575.0,1097038.0,241932.0,118420,9.948289,1908.1147
4,Afghanistan,2006,25424100.0,1120880.0,245586.0,116622,10.622766,1929.7239


CSV guardado: U3_1_modelos_ml/data/dataset_health_gdp_demografia_final.csv (4,244 filas)


## 2. Incorporar el objetivo para modelado
El archivo de modelado conserva las seis predictoras y `life_expectancy` en años, además
de la llave entidad-año. Se mantiene `code` en un archivo auxiliar para excluir agregados
al entrenar; no se usan nombres, códigos ni años como predictoras.

In [3]:
modelado = integrado.merge(vida.drop(columns="code"), on=["entity", "year"], validate="one_to_one")
modelado = modelado.dropna(subset=[*columnas[2:], "life_expectancy"]).copy()
modelado = solo_paises(modelado).sort_values(["entity", "year"]).reset_index(drop=True)
columnas_modelo = ["entity", "year", "population", "births", "deaths", "child_deaths_under_5", "percentage_health", "gdp", "life_expectancy"]
guardar_tabla(modelado[columnas_modelo], "dataset_7variables_2000_2022.csv", U3 / "data/raw")
guardar_tabla(modelado[["entity", "code"]].drop_duplicates(), "codigos_modelado.csv")
display(modelado[columnas_modelo].describe().round(2))
cobertura = modelado.groupby("year").size().rename("entidades").reset_index()
display(cobertura)
guardar_tabla(cobertura, "cobertura_dataset_integrado.csv")

CSV guardado: U3_1_modelos_ml/data/raw/dataset_7variables_2000_2022.csv (4,221 filas)
CSV guardado: U3_1_modelos_ml/data/processed/codigos_modelado.csv (186 filas)


,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp,life_expectancy
count,4221.00,4.221000e+03,4221.00,4221.00,4221.00,4221.00,4221.00,4221.00
mean,2011.08,3.784338e+07,741609.90,297812.78,38404.30,6.20,21745.13,70.05
std,6.62,1.408400e+08,2452894.17,1017412.06,150574.09,2.83,23589.20,8.73
min,2000.00,9.566000e+03,197.00,66.00,0.00,1.22,711.98,14.67
25%,2005.00,1.907122e+06,31542.00,14039.00,318.00,4.12,4793.56,64.46
50%,2011.00,7.299003e+06,136888.00,57031.00,2786.00,5.64,13135.72,71.46
75%,2017.00,2.539999e+07,605872.00,198656.00,22257.00,7.95,30602.11,76.49
max,2022.00,1.426437e+09,29324930.00,13093783.00,2636156.00,24.46,149025.11,85.71


,year,entidades
0,2000,179
1,2001,179
2,2002,180
3,2003,182
4,2004,182
...,...,...
18,2018,186
19,2019,186
20,2020,186
21,2021,186


CSV guardado: U3_1_modelos_ml/data/processed/cobertura_dataset_integrado.csv (23 filas)


## 3. Respuestas y límites
**¿Por qué disminuyen las filas?** Las fuentes no cubren todas las mismas entidades y años;
se retiene su intersección con datos completos. **¿Qué representan las columnas nuevas?**
`percentage_health` es gasto corriente en salud como porcentaje del PIB; `gdp` es PIB por
persona en dólares internacionales constantes. **¿Se imputó?** No. Excluir faltantes puede
sesgar la muestra hacia países mejor documentados y debe reconocerse al interpretar resultados.
No se incorporan proyecciones como objetivos de entrenamiento.

In [4]:
display(Markdown(f"**Entrega:** {len(tarea):,} registros con las ocho columnas solicitadas. "
f"El archivo de modelado contiene {len(modelado):,} registros completos de {modelado.entity.nunique()} entidades "
f"con códigos de tres letras, en {modelado.year.min()}-{modelado.year.max()}."))

**Entrega:** 4,244 registros con las ocho columnas solicitadas. El archivo de modelado contiene 4,221 registros completos de 186 entidades con códigos de tres letras, en 2000-2022.